<div style="background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); color: white; padding: 25px; text-align: center; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.2); margin: 20px 0;">

# Demo.1 - MediaPipe 手部关键点检测

本 Notebook 演示如何使用 **MediaPipe Tasks API** 实时检测手部 21 个关键点（Landmarks），并以彩色发光效果绘制骨架。

</div>

<div style="background-color: #eef6ff; border-left: 4px solid #2563eb; padding: 15px; border-radius: 4px; margin-top: 12px;">

## 本节你将学会

1. 理解 MediaPipe 返回的 21 个关键点分别代表什么。
2. 把归一化坐标转换为屏幕上的像素坐标。
3. 在实时视频中绘制手部骨架、左右手标签和 FPS 信息。
4. 观察不同手势下，关键点位置如何随动作连续变化。

**建议课堂观察**：张开手掌、握拳、左右摆手，看看哪些关键点最稳定，哪些关键点变化最大。

</div>

### 课堂观察任务

请不要把这个实验只当成“把骨架画出来”。运行前先带着下面 3 个观察任务：

1. 哪些关键点最稳定：手腕、掌骨关节，还是指尖？
2. 哪些动作最容易让关键点抖动：快速摆手、握拳，还是半遮挡？
3. 画面是镜像显示时，为什么看到的“左手”与模型原始标签可能相反？

建议同学边实验边记录：
- 静止张开手掌时，关键点是否连续稳定。
- 快速移动时，关键点是否会短暂漂移或丢失。
- 手指弯曲时，哪些关节点变化最明显。


<div style="background-color: #e7f3ff; border-left: 4px solid #2196F3; padding: 15px; text-align: border-radius: 4px;">



## 手部 21 个关键点索引
- 0: 手腕（Wrist）
- 1–4: 拇指（Thumb）
- 5–8: 食指（Index）
- 9–12: 中指（Middle）
- 13–16: 无名指（Ring）
- 17–20: 小指（Pinky）
</div>




<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 0.环境安装

首次运行，请执行以下安装命令

</div>



### 弹窗显示的重要说明

如果你希望使用 `cv2.imshow(...)` 打开 OpenCV 弹窗，当前环境里**不能同时混装多个 OpenCV 版本或 headless 版本**。

当前这类 MediaPipe 教程如果装了 `opencv-python-headless`，会导致 OpenCV 变成 **GUI: NONE**，此时背景切换虽然能计算，但窗口无法弹出。

需要先卸载冲突包，再安装带 GUI 的版本。安装完成后，请**重启内核**再继续运行。


In [1]:
# %pip uninstall -y opencv-python-headless opencv-python opencv-contrib-python opencv-contrib-python-headless
%pip install mediapipe opencv-python numpy

Note: you may need to restart the kernel to use updated packages.


### 安装后必须执行这一步

如果上面的安装单元执行完成，请先**重启内核**，再从“导入依赖库”开始重新运行。

原因是：
- 当前 notebook 内核可能还保留着旧的 `cv2` 模块。
- 不重启的话，即使你已经安装了带 GUI 的 OpenCV，`cv2.imshow` 仍可能继续报错。


<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 1. 导入依赖库

</div>


In [2]:
import time
from pathlib import Path

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision

print("依赖库导入成功 ✓")

依赖库导入成功 ✓



<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 2. 常量配置与模型读取

手部连接关系（HAND_CONNECTIONS）

每个元组 `(a, b)` 表示关键点 `a` 与 `b` 之间存在骨骼连接。
共 21 条连接线，覆盖拇指 4 段、四指各 4 段、手掌基部 1 段。

</div>



In [4]:
WINDOW_NAME = "Demo.1 - MediaPipe Hands Landmarks"
MODEL_PATH = Path("models/hand_landmarker.task")

# 手部骨骼连接关系：(起点索引, 终点索引)
HAND_CONNECTIONS = frozenset([
    # 拇指
    (0, 1), (1, 2), (2, 3), (3, 4),
    # 食指
    (0, 5), (5, 6), (6, 7), (7, 8),
    # 中指
    (5, 9), (9, 10), (10, 11), (11, 12),
    # 无名指
    (9, 13), (13, 14), (14, 15), (15, 16),
    # 小指
    (13, 17), (17, 18), (18, 19), (19, 20),
    # 手掌基部
    (0, 17),
])

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"未找到手部模型文件: {MODEL_PATH}")

print(f"骨骼连接数: {len(HAND_CONNECTIONS)}")
print(f"模型路径: {MODEL_PATH} ✓")
print("课堂提示：如果镜像画面里看到的是左手，模型原始结果可能会标成 Right，需要结合镜像再理解。")

骨骼连接数: 21
模型路径: models\hand_landmarker.task ✓
课堂提示：如果镜像画面里看到的是左手，模型原始结果可能会标成 Right，需要结合镜像再理解。



<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 3. 坐标转换

MediaPipe 返回的关键点坐标是 **归一化坐标**（值域 0~1），需要乘以图像宽/高才能得到像素坐标。

$$pixel\_x = landmark.x \times width, \quad pixel\_y = landmark.y \times height$$

</div>

In [5]:
def landmark_to_pixel(landmark, width, height):
    """将归一化关键点坐标（0~1）转换为像素坐标。"""
    return int(landmark.x * width), int(landmark.y * height)

# 示例：归一化 (0.5, 0.5) 在 640×480 图像中 → 像素 (320, 240)
#   0.5 × 640 = 320（水平中心）
#   0.5 × 480 = 240（垂直中心）
print("归一化 (0.5, 0.5) 在 640×480 图像中 → 像素 (320, 240)")

归一化 (0.5, 0.5) 在 640×480 图像中 → 像素 (320, 240)



<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 4. 绘图工具函数

- `draw_glow_point`：绘制发光关键点（光晕 → 白边 → 彩色核心）
- `add_panel`：在画面左上角叠加半透明 FPS 信息面板

</div>




In [6]:
def draw_glow_point(image, center, radius, color):
    """绘制发光圆点：光晕 → 白边 → 彩色核心。"""
    cv2.circle(image, center, radius + 6, color, -1, cv2.LINE_AA)
    cv2.circle(image, center, radius + 2, (255, 255, 255), -1, cv2.LINE_AA)
    cv2.circle(image, center, radius, color, -1, cv2.LINE_AA)


def add_panel(image, title, fps_text):
    """在图像左上角绘制半透明信息面板。"""
    overlay = image.copy()
    cv2.rectangle(overlay, (18, 18), (420, 138), (20, 28, 48), -1)
    cv2.addWeighted(overlay, 0.42, image, 0.58, 0, image)
    cv2.putText(image, title, (32, 52),
                cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 240, 220), 2, cv2.LINE_AA)
    cv2.putText(image, fps_text, (32, 88),
                cv2.FONT_HERSHEY_SIMPLEX, 0.72, (120, 240, 255), 2, cv2.LINE_AA)
    cv2.putText(image, "Press Q or ESC to quit", (32, 124),
                cv2.FONT_HERSHEY_SIMPLEX, 0.62, (220, 230, 240), 2, cv2.LINE_AA)


<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 5. 手部骨架绘制

指尖（索引 4/8/12/16/20）画橙色发光大圆，其余关节画黄色小圆，连线统一用灰色。

</div>


In [7]:
def draw_hand_landmarks(image, hand_landmarks, hand_label):
    """绘制手部骨架和关键点。"""
    h, w = image.shape[:2]
    points = [landmark_to_pixel(lm, w, h) for lm in hand_landmarks]

    # 绘制骨骼连线（灰色）
    for a, b in HAND_CONNECTIONS:
        cv2.line(image, points[a], points[b], (180, 180, 180), 2, cv2.LINE_AA)

    # 绘制关键点：指尖橙色大圆，其余黄色小圆
    for idx, pt in enumerate(points):
        if idx in {4, 8, 12, 16, 20}:      # 指尖
            draw_glow_point(image, pt, 7, (0, 165, 255))
        else:                                # 其他关节
            draw_glow_point(image, pt, 5, (0, 255, 255))

    # 手腕处标注左右手
    wrist = points[0]
    cv2.putText(image, hand_label, (wrist[0] - 20, wrist[1] - 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 6. 主程序：实时视频处理循环

### 关键步骤
1. **摄像头采集**：`cv2.VideoCapture(0)` 打开默认摄像头。
2. **图像预处理**：BGR → RGB，封装为 `mp.Image`。
3. **结果解析**：`results.hand_landmarks` 是检测到的每只手的关键点列表。
4. **可视化绘制**：关键点、骨架、左右手标签与 FPS 同步叠加。

### 课堂上建议同学重点观察
1. 张开手掌和握拳时，哪些关键点移动幅度最大？
2. 手快速移动时，关键点是否仍然稳定跟踪？
3. 镜像显示下的 Left / Right 标签为什么需要反向理解？

**运行后**：会弹出 OpenCV 窗口，按 `Q` 退出。

</div>

In [9]:
def main():
    from IPython.display import clear_output, display
    from PIL import Image

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("无法打开摄像头 0。")

    def highgui_available():
        try:
            cv2.namedWindow('highgui_test', cv2.WINDOW_NORMAL)
            cv2.destroyWindow('highgui_test')
            return True
        except cv2.error:
            return False

    use_highgui = highgui_available()
    max_preview_frames = 60
    frame_counter = 0
    prev_time = time.time()

    if use_highgui:
        print("已启用 OpenCV 窗口模式，按 Q 或 ESC 退出。")
    else:
        print("当前环境不支持 cv2.imshow，已切换为 notebook 内联预览模式。")
        print(f"内联模式将自动预览 {max_preview_frames} 帧。")

    options = mp_vision.HandLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=mp_vision.RunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.6,
        min_hand_presence_confidence=0.6,
        min_tracking_confidence=0.6,
    )
    with mp_vision.HandLandmarker.create_from_options(options) as landmarker:
        while True:
            success, frame = cap.read()
            if not success:
                print("读取摄像头帧失败。")
                break

            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int(time.time() * 1000)
            results = landmarker.detect_for_video(mp_image, timestamp_ms)

            canvas = frame.copy()
            if results.hand_landmarks:
                for idx, hand_landmarks in enumerate(results.hand_landmarks):
                    hand_label = "Hand"
                    if idx < len(results.handedness):
                        raw_label = results.handedness[idx][0].category_name
                        hand_label = "Left" if raw_label == "Right" else "Right"
                    draw_hand_landmarks(canvas, hand_landmarks, hand_label)

            current_time = time.time()
            fps = 1.0 / max(current_time - prev_time, 1e-6)
            prev_time = current_time
            add_panel(canvas, "MediaPipe Hands", f"FPS: {fps:.1f}")

            if use_highgui:
                cv2.imshow(WINDOW_NAME, canvas)
                key = cv2.waitKey(1) & 0xFF
                if key in (ord("q"), 27):
                    break
            else:
                canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
                clear_output(wait=True)
                display(Image.fromarray(canvas_rgb))
                time.sleep(0.05)
                frame_counter += 1
                if frame_counter >= max_preview_frames:
                    print("内联预览已结束。若要继续观察，请重新运行本单元。")
                    break

    cap.release()
    if use_highgui:
        try:
            cv2.destroyAllWindows()
            cv2.waitKey(1)
        except cv2.error:
            pass


main()

已启用 OpenCV 窗口模式，按 Q 或 ESC 退出。


### 运行前，先预测一下会看到什么

在点击运行前，可以先做两个预测：

1. 如果手掌完全张开，哪几个指尖关键点会最容易区分？
2. 如果握拳或部分手指被遮挡，模型最可能在哪些位置画不稳？

运行时建议按这个顺序测试：
- 先用静止手掌确认 21 个关键点和骨架连线都能稳定显示。
- 再做握拳、张开、左右摆动，比较关键点轨迹变化。
- 最后尝试把手靠近镜头或快速移动，观察 FPS 与稳定性的关系。

### 结果解读与排错提示

如果课堂演示时效果不理想，可以优先检查这几项：

1. 画面太暗或手离镜头太远，关键点会抖动或直接丢失。
2. 手掌部分被遮挡时，指尖和关节点更容易检测失败。
3. 如果摄像头打不开，通常是权限、占用，或设备编号不是 0。

建议同学做一个小实验：分别比较“静止手掌”“快速摆手”“半遮挡手掌”三种情况下的关键点稳定性。